# Day H2 — 표본 수와 분할: 몇 명이면 충분하고, 어떻게 나누는가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — Riley 기준으로 최소 표본 수 계산하기

기대 성능을 모를 때의 보수적 출발점(Nagelkerke R² 0.15)을 가정하고, 후보 변수 수와 유병률을 바꿔 봅니다.

In [ ]:
import numpy as np

def riley_binary(p, prev, r2_nagelkerke=0.15, shrink=0.9, delta=0.05, margin=0.05):
    """Riley 등(2020)의 세 기준으로 이진 결과 예측 모델의 최소 표본 수를 계산한다."""
    lnL0 = prev * np.log(prev) + (1 - prev) * np.log(1 - prev)   # 1명당 null 로그우도
    max_r2cs = 1 - np.exp(2 * lnL0)                               # 이 유병률에서 R2_CS가 가질 수 있는 최댓값
    r2cs = r2_nagelkerke * max_r2cs                               # 기대 성능을 R2_CS 단위로
    n1 = p / ((shrink - 1) * np.log(1 - r2cs / shrink))           # 기준 1: 축소 계수 0.9 이상
    s2 = r2cs / (r2cs + delta * max_r2cs)
    n2 = p / ((s2 - 1) * np.log(1 - r2cs / s2))                   # 기준 2: 겉보기·보정 R2 차이 0.05 이하
    n3 = (1.96 / margin) ** 2 * prev * (1 - prev)                 # 기준 3: 전체 위험을 ±0.05로 추정
    n = int(np.ceil(max(n1, n2, n3)))
    return int(np.ceil(n1)), int(np.ceil(n2)), int(np.ceil(n3)), n, n * prev, n * prev / p

print("후보 변수  유병률 | 기준1  기준2  기준3 | 최소 n  사건 수  EPP")
for p, prev in [(10, 0.30), (10, 0.10), (20, 0.10), (10, 0.03)]:
    n1, n2, n3, n, ev, epp = riley_binary(p, prev)
    print(f"{p:>6d}    {prev:>5.2f} | {n1:>5d}  {n2:>5d}  {n3:>5d} | {n:>6d}  {ev:>7.0f}  {epp:>4.1f}")

### 예시 2 — 표본이 작으면 무엇이 무너지나

후보 변수 20개 중 실제로 관련 있는 것은 4개인 가상 세계에서, 개발 표본 크기를 바꿔 같은 연구를 50번씩 반복합니다.

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(2)
p = 20                                   # 후보 변수 20개, 그중 실제로 관련 있는 것은 4개
beta = np.r_[0.8, 0.6, 0.5, 0.4, np.zeros(p - 4)]

def make(n):
    X = rng.normal(size=(n, p))
    y = rng.random(n) < 1 / (1 + np.exp(-(-2.2 + X @ beta)))   # 유병률 약 15%
    return X, y.astype(int)

X_test, y_test = make(20000)             # "모집단"을 대신하는 큰 시험 세트
print("개발 n  사건 수  EPP  | 겉보기 AUC  시험 AUC  보정 기울기")
for n in [150, 400, 1500, 6000]:
    res = []
    for rep in range(50):                # 같은 크기의 연구를 50번 반복
        X, y = make(n)
        m = LogisticRegression(C=1e6, max_iter=2000).fit(X, y)   # 사실상 벌점 없는 최대우도
        lp = m.decision_function(X_test)
        slope = sm.Logit(y_test, sm.add_constant(lp)).fit(disp=0).params[1]
        res.append((y.sum(), roc_auc_score(y, m.decision_function(X)), roc_auc_score(y_test, lp), slope))
    ev, app, test, slope = np.mean(res, axis=0)
    print(f"{n:>6d}  {ev:>6.0f}  {ev / p:>4.1f} |   {app:.3f}      {test:.3f}      {slope:.2f}")

### 예시 3 — 기관 하나씩 빼기

기관 4곳, 한 사람당 기록 3개인 가상 자료입니다. 기관 C와 D는 기기 증폭률이 다르고, 기관마다 유병률이 다릅니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold

rng = np.random.default_rng(3)
rows = []
pid = 0
for site, (n_people, prev, gain) in {"A": (120, 0.5, 1.0), "B": (120, 0.4, 1.0),
                                     "C": (100, 0.3, 1.3), "D": (80, 0.2, 0.8)}.items():
    for _ in range(n_people):
        pid += 1
        case = rng.random() < prev
        person = rng.normal(0, 1)                     # 사람마다 고유한 기준값
        for rec in range(3):                          # 한 사람당 기록 3개
            tremor = gain * (0.8 * case + person + rng.normal(0, 0.5))   # 기기마다 증폭률(gain)이 다르다
            sway = 0.5 * case + 0.6 * person + rng.normal(0, 1)
            rows.append(dict(pid=pid, site=site, case=int(case), tremor=tremor, sway=sway))
df = pd.DataFrame(rows)
X, y = df[["tremor", "sway"]].values, df.case.values
print(df.groupby("site").agg(사람=("pid", "nunique"), 기록=("pid", "size"), 유병률=("case", "mean")).round(2))

def cv_auc(splitter, groups=None):
    scores = []
    for tr, te in splitter.split(X, y, groups):
        m = LogisticRegression().fit(X[tr], y[tr])
        scores.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return np.array(scores)

print("\n기록 단위 무작위 5-fold :", cv_auc(StratifiedKFold(5, shuffle=True, random_state=0)).mean().round(3))
print("참가자 단위 5-fold      :", cv_auc(GroupKFold(5), df.pid).mean().round(3))
print("기관 하나씩 빼기:")
for site in sorted(df.site.unique()):          # 기관 하나씩 빼기(internal-external CV)
    tr, te = df.site != site, df.site == site
    m = LogisticRegression().fit(X[tr], y[tr])
    p = m.predict_proba(X[te])[:, 1]
    print(f"  {site}를 빼고 개발 → {site}에서 검증: AUC {roc_auc_score(y[te], p):.3f} | 관찰 {y[te].mean():.2f} vs 예측 평균 {p.mean():.2f}")